# Lesson 11A: Model-Based Reinforcement Learning - Theory

## Introduction

Model-free RL learns directly from experience. Model-based RL learns a **model** of the environment dynamics, then plans using that model.

- **World model**: p(s' | s, a) — what happens next?
- **Planning**: Use the model to simulate trajectories and choose actions
- **Dyna**: Blend real experience with simulated experience
- **Trade-off**: Model learning cost vs. sample efficiency gain

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Part 1: Why Learn a Model?

### Sample Efficiency

Model-free RL requires many environment interactions. Model-based RL:
1. Learn p(s' | s, a) from real experience
2. Use the model to generate synthetic experience ("imagination")
3. Train policy/value on both real + imagined data

**Result**: Fewer real environment steps needed.

### Planning

With a model, you can plan ahead:
$$V(s) = \max_a \mathbb{E}[r(s,a) + \gamma V(s')]$$

Where s' is drawn from the learned model p(s'|s,a).

## Part 2: Dyna: Integrating Planning and Learning

### The Dyna Architecture

1. **Act**: Take action in real environment, get (s, a, r, s')
2. **Learn model**: Update model p(s'|s,a) and reward r(s,a)
3. **Learn value**: Update V(s) from real experience
4. **Plan**: Simulate K trajectories using the model, update V again

### Pseudocode

```
repeat:
  # Real experience
  (s, a, r, s') <- environment.step()
  
  # Model learning
  model.update(s, a, s', r)
  
  # Direct RL
  V(s) <- r + gamma * V(s')
  
  # Simulated planning (Dyna)
  for _ in range(planning_steps):
    s_sim ~ visited_states  # replay from buffer
    a_sim ~ policy(s_sim)
    (s'_sim, r_sim) ~ model(s_sim, a_sim)  # imagination
    V(s_sim) <- r_sim + gamma * V(s'_sim)
```

**Key insight**: Every real step generates multiple value updates (planning).

## Part 3: World Models and Latent Planning

### Beyond Tabular Models

In large state spaces, learning p(s'|s,a) directly is intractable. Use:

1. **Learned embeddings**: Encoder φ(s) → z (latent)
2. **Latent model**: p(z'|z,a) — much smaller
3. **Decoder**: ψ(z) → reconstructed state
4. **Plan in latent space**: Cheaper than planning in pixel space

### Architectures

- **World Models (Ha & Schmidhuber, 2018)**: VAE + RNN for video generation
- **PlaNet (Hafner et al., 2019)**: Deterministic latent model
- **Dreamer (Hafner et al., 2020)**: Learn policy via latent imagination

## Part 4: Planning Methods

### Model Predictive Control (MPC)

At each step, solve:
$$\max_{a_1, ..., a_H} \sum_{t=1}^H \gamma^t r_t$$

Subject to: $s_{t+1} = f(s_t, a_t)$ (learned dynamics)

**Approach**: Cross-entropy method, CEM (sample actions, fit distribution to best ones).

### Monte Carlo Tree Search (MCTS)

Build a search tree by simulating rollouts:
1. **Select**: Traverse tree via UCB until leaf
2. **Expand**: Add child node
3. **Simulate**: Rollout to horizon with random actions
4. **Backup**: Propagate return up the tree

Used in AlphaGo, AlphaZero (deterministic planning + neural networks).

In [ ]:
# Conceptual Dyna with tabular model
class DynaAgent:
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.99, planning_steps=10):
        self.n_states = n_states
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps
        
        # Model: transition probabilities and rewards
        self.model_p = np.ones((n_states, n_actions, n_states)) / n_states
        self.model_r = np.zeros((n_states, n_actions))
        
        # Value function
        self.V = np.zeros(n_states)
        
        # Visited state-action pairs (for planning)
        self.visited = set()
    
    def update_model(self, s, a, r, s_next):
        """Update model from real experience."""
        self.model_r[s, a] = (1 - self.alpha) * self.model_r[s, a] + self.alpha * r
        self.model_p[s, a, s_next] += self.alpha
        self.model_p[s, a] /= self.model_p[s, a].sum()
        self.visited.add((s, a))
    
    def direct_rl(self, s, r, s_next):
        """Direct temporal difference update."""
        self.V[s] = self.V[s] + self.alpha * (r + self.gamma * self.V[s_next] - self.V[s])
    
    def planning(self):
        """Dyna planning: simulate and improve."""
        for _ in range(self.planning_steps):
            if not self.visited:
                continue
            
            s, a = list(self.visited)[np.random.randint(len(self.visited))]
            
            # Simulate: sample from model
            r = self.model_r[s, a]
            s_next = np.random.choice(self.n_states, p=self.model_p[s, a])
            
            # Update value
            self.V[s] = self.V[s] + self.alpha * (r + self.gamma * self.V[s_next] - self.V[s])
    
    def step(self, s, a, r, s_next):
        """One Dyna step: learn model, direct RL, planning."""
        self.update_model(s, a, r, s_next)
        self.direct_rl(s, r, s_next)
        self.planning()

print("DynaAgent: defined")

## Part 5: Challenges and Trade-offs

### Model Error Accumulation

Errors in the learned model compound over long rollouts:
$$\text{error at step } t \approx t \times \text{error per step}$$

**Fix**: Plan over shorter horizons, or use uncertainty (ensemble models).

### Sample vs. Compute

- **Model-free**: Few assumptions, needs many environment samples
- **Model-based**: More computation (learning + planning), fewer samples

**Hybrid**: Dreamer, MuZero — learn compact models, plan efficiently.

### When to Use Model-Based RL

✅ **Good for**:
- Sample efficiency critical (real robots, expensive simulations)
- Offline RL (no online environment access)
- Curiosity-driven exploration

❌ **Poor for**:
- Cheap simulation (compute > samples)
- Stochastic environments (hard to model)
- High-dimensional observations (expensive to model)

## Key Concepts

1. **World model**: p(s'|s,a) + r(s,a) — learn environment dynamics
2. **Dyna**: Integrate direct RL + planning in one loop
3. **Latent models**: Learn compact representations, plan in latent space
4. **Planning horizons**: Short horizons reduce error accumulation
5. **MPC & MCTS**: Deterministic planning methods

Next: Implement Dyna in PyTorch on a tabular environment, then latent planning.